# Fine-tuning (S5.17/S5.18)

Unsloth + TRL LoRA fine-tuning for the Standard (Qwen3-8B) and Lite (Qwen3-4B) models. Checkpoints land on Drive and training is resumable - never assume one uninterrupted session.

In [ ]:
import os
os.environ["HAVYN_DRIVE"] = env.drive_dir or "/content/havyn-artifacts"
os.environ["HAVYN_REPO"] = "/content/MCP-Packet-Tracer"
print("HAVYN_DRIVE =", os.environ["HAVYN_DRIVE"])

In [ ]:
# Havyn Stage 5 - environment setup and verification gate.
# Every notebook runs this first: no notebook trains or generates data on an
# unverified environment.
import sys, shutil, os
if not shutil.which("git"):
    raise RuntimeError("git missing")
!git clone --depth 1 https://github.com/HamzaNinjaCoder1/MCP-Packet-Tracer.git /content/MCP-Packet-Tracer 2>/dev/null || git -C /content/MCP-Packet-Tracer pull --ff-only
import sys
sys.path.insert(0, "/content/MCP-Packet-Tracer/havyn-stage5/scripts")
os.chdir("/content/MCP-Packet-Tracer")
from colab_setup import setup, save_report
env = setup(run_tests=True, use_drive=True)
assert not env.errors, f"environment verification failed: {env.errors}"
assert env.tests_passed, "offline suite failed - STOP, do not continue"
from env_report import build_report, recommend_config
report = build_report(repo="/content/MCP-Packet-Tracer", dataset_version="v0")
import json, pathlib
out_dir = pathlib.Path(env.drive_dir or "/content/havyn-artifacts")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "env-report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(recommend_config(report.get("gpu", {})), indent=2))
print(json.dumps(report, indent=2)[:2000])


In [ ]:
!pip install -q unsloth
# No secrets in the notebook: read tokens via Colab secrets only.
#   from google.colab import userdata; os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
# The dataset must be frozen and byte-identical (S5.16); train_lora
# refuses otherwise. Copy the frozen splits to Drive, then train.
!python havyn-stage5/scripts/freeze_dataset.py havyn-stage5/datasets/havyn-train-v1 --check
!mkdir -p "$HAVYN_DRIVE/datasets" && rm -rf "$HAVYN_DRIVE/datasets/havyn-train-v1" && cp -r havyn-stage5/datasets/havyn-train-v1 "$HAVYN_DRIVE/datasets/"
!PYTHONPATH=havyn-stage5/scripts:. python havyn-stage5/scripts/train_lora.py \n  --config havyn-stage5/configs/train-standard.yaml --resume \n  --dataset-dir "$HAVYN_DRIVE/datasets/havyn-train-v1" \n  --output-dir "$HAVYN_DRIVE/checkpoints/standard"

In [ ]:
# Lite model: same frozen dataset, smaller base.
!PYTHONPATH=havyn-stage5/scripts:. python havyn-stage5/scripts/train_lora.py \n  --config havyn-stage5/configs/train-lite.yaml --resume \n  --dataset-dir "$HAVYN_DRIVE/datasets/havyn-train-v1" \n  --output-dir "$HAVYN_DRIVE/checkpoints/lite"